In [1]:
# 기본 설정

from supabase import create_client, Client

TARGET_DB="content_DB"
MIGRATION_DB="practice_DB"
UPDATE_SIZE=1000
SUPABASE_KEY='sb_publishable_DVlQhSuIouv53mYz9NAFSQ_WBuLqavM'
SUPABASE_URL='https://jjnlqyxgxtzxeirksgqq.supabase.co'

supabase:Client=create_client(SUPABASE_URL, SUPABASE_KEY)

In [2]:
def get_target_numbers() -> tuple[int, int]:
    """Target DB와 Migration DB의 최신 게시글 번호를 반환합니다."""

    try:
        start_response = (
            supabase.table(TARGET_DB)
            .select("post_num")
            .order("post_num", desc=True)
            .limit(1)
            .maybe_single()
            .execute()
        )

        start_postnum = (
            start_response.data["post_num"]
            if start_response.data
            else 0
        )

    except Exception as e:
        print(f"Target DB의 최신 게시글 번호 조회 실패: {e}")
        start_postnum= 0

    try:
        target_response = (
            supabase.table(MIGRATION_DB)
            .select("post_num")
            .order("post_num", desc=True)
            .limit(1)
            .maybe_single()
            .execute()
        )

        target_postnum = (
            target_response.data["post_num"]
            if target_response.data
            else 0
        )

    except Exception as e:
        print(f"Migration DB의 최신 게시글 번호 조회 실패: {e}")
        target_postnum = 0

    return start_postnum, target_postnum

In [12]:
response=(supabase.table('practice_DB').select('post_num').order('post_num').execute())
target_numlist=[]
for postnum in response.data:
    target_numlist.append(postnum['post_num'])
    
target_numlist=set(target_numlist)

In [19]:
response_b=(supabase.table('content_DB').select('post_num').order('post_num').execute())
migration_numlist=[]
for postnum in response_b.data:
    migration_numlist.append(postnum['post_num'])

migration_numlist=set(migration_numlist)

In [3]:
start_postnum, target_postnum = get_target_numbers()

print(f"마이그레이션 시작점: {start_postnum}")
print(f"마이그레이션 종료지점: {target_postnum}")

마이그레이션 시작점: 17694271
마이그레이션 종료지점: 17701734


In [ ]:
chunck_unit=target_postnum-start_postnum

3199

In [4]:
while True:
    response=(
        supabase
        .table('practice_DB')
        .select('post_num, date')
        .gt('post_num',start_postnum)
        .lte("post_num", target_postnum)
        .order('post_num')
        .limit(UPDATE_SIZE)
    .execute())
    
    posts=response.data or []
    
    if not posts:
        print('모든 게시글 적재 완료')
        break
    
    
    rows=[
        {'post_num':row["post_num"],
        'created_at':row["date"]}
        for row in posts]
    
    (supabase
     .table('content_DB')
     .upsert(
         rows,
         on_conflict='post_num',
         ignore_duplicates=True
     ).execute()
     )
    start_postnum=posts[-1]['post_num']
    
    print(
        f'이번 배치 {len(posts)}건'
        f'다음 조회 기준 post_num: {start_postnum:,}'
    )
        

이번 배치 1000건다음 조회 기준 post_num: 17,699,959
이번 배치 1000건다음 조회 기준 post_num: 17,701,042
이번 배치 656건다음 조회 기준 post_num: 17,701,734
모든 게시글 적재 완료
